In [1]:
import pandas as pd
import numpy as np

PARQUET_PATH = "statcast_2021_2025.parquet"

raw = pd.read_parquet(PARQUET_PATH)
print("loaded raw:", raw.shape)
print("cols:", len(raw.columns))

loaded raw: (3846144, 119)
cols: 119


In [2]:
# =========================
# Build PA-labeled dataset
# =========================

# find pitch order column
sort_col = "pitch_number" if "pitch_number" in raw.columns else ("pitch_num" if "pitch_num" in raw.columns else None)
if sort_col is None:
    raise ValueError("Need 'pitch_number' or 'pitch_num' to order pitches within PA.")

needed = [
    "game_pk","at_bat_number",sort_col,
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"
]

missing = [c for c in needed if c not in raw.columns]
if missing:
    raise ValueError(f"Missing columns in parquet: {missing}")

tmp = raw[needed].copy()

# runners
tmp["runner_1b"] = tmp["on_1b"].notna().astype(np.int8)
tmp["runner_2b"] = tmp["on_2b"].notna().astype(np.int8)
tmp["runner_3b"] = tmp["on_3b"].notna().astype(np.int8)
tmp.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# run diff (batting - fielding)
tmp["run_diff"] = (tmp["bat_score"].fillna(0) - tmp["fld_score"].fillna(0)).astype(np.int16)
tmp.drop(columns=["bat_score","fld_score"], inplace=True)

# sort and get final pitch per PA
tmp = tmp.sort_values(["game_pk","at_bat_number",sort_col])
last_pitch = tmp.groupby(["game_pk","at_bat_number"], as_index=False).tail(1).copy()

def map_pa_outcome(events, des):
    ev = None if pd.isna(events) else str(events)
    d = "" if pd.isna(des) else str(des).lower()

    if ev == "single": return "single"
    if ev == "double": return "double"
    if ev == "triple": return "triple"
    if ev == "home_run": return "hr"

    if ev in ["walk","intent_walk","hit_by_pitch"]:
        return "walk"

    if "strikeout" in d or ev in ["strikeout","strikeout_double_play"]:
        return "strikeout"

    if ev is not None:
        return "out"

    return "none"

last_pitch["pa_outcome"] = [map_pa_outcome(e, d) for e, d in zip(last_pitch["events"], last_pitch["des"])]
last_pitch = last_pitch[last_pitch["pa_outcome"] != "none"][["game_pk","at_bat_number","pa_outcome"]].copy()

# attach PA outcome to every pitch-state row
tmp = tmp.merge(last_pitch, on=["game_pk","at_bat_number"], how="inner")
tmp.drop(columns=["events","des"], inplace=True)

# keep only columns we’ll model with
features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

# coerce numeric
for c in ["release_speed","release_spin_rate","pfx_x","pfx_z","zone"]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

tmp = tmp.dropna(subset=["pitch_type","stand","p_throws","release_speed","pfx_x","pfx_z","zone"]).copy()

df = tmp[features + ["pa_outcome"]].copy()

print("modeling df:", df.shape)
print("pa_outcome dist (%):")
print(df["pa_outcome"].value_counts(normalize=True).mul(100).round(2))

modeling df: (3741396, 19)
pa_outcome dist (%):
pa_outcome
out          39.80
strikeout    28.31
walk         13.03
single       12.16
double        3.72
hr            2.65
triple        0.33
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier

X = df[features].copy()
y = df["pa_outcome"].copy()

# categorical columns as strings (GPU safety)
cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    eval_metric="MultiClass",
    verbose=100,
    task_type="GPU",
    devices="0"
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

probs = model.predict_proba(X_test)
print("logloss:", log_loss(y_test, probs, labels=model.classes_))
print("classes:", list(model.classes_))

0:	learn: 1.8900846	test: 1.8901197	best: 1.8901197 (0)	total: 3.13s	remaining: 1h 2m 27s
100:	learn: 1.3859074	test: 1.3855144	best: 1.3855144 (100)	total: 12.8s	remaining: 2m 19s
200:	learn: 1.3791022	test: 1.3793661	best: 1.3793661 (200)	total: 21.9s	remaining: 1m 48s
300:	learn: 1.3753550	test: 1.3765873	best: 1.3765873 (300)	total: 30.6s	remaining: 1m 31s
400:	learn: 1.3726570	test: 1.3748487	best: 1.3748487 (400)	total: 39.3s	remaining: 1m 18s
500:	learn: 1.3700120	test: 1.3732397	best: 1.3732397 (500)	total: 48.2s	remaining: 1m 7s
600:	learn: 1.3678505	test: 1.3719881	best: 1.3719881 (600)	total: 57.1s	remaining: 56.9s
700:	learn: 1.3656894	test: 1.3707968	best: 1.3707968 (700)	total: 1m 6s	remaining: 47s
800:	learn: 1.3633966	test: 1.3695128	best: 1.3695128 (800)	total: 1m 15s	remaining: 37.7s
900:	learn: 1.3613153	test: 1.3684029	best: 1.3684029 (900)	total: 1m 24s	remaining: 28.1s
1000:	learn: 1.3592440	test: 1.3673002	best: 1.3673002 (1000)	total: 1m 33s	remaining: 18.6s
110

In [4]:
from pybaseball import playerid_lookup

# --- name -> mlbam id ---
_player_cache = {}

def mlbam_id_from_name(name):
    name = name.strip()
    if name in _player_cache:
        return _player_cache[name]
    parts = name.split()
    if len(parts) < 2:
        raise ValueError("Use 'First Last' for player name.")
    first, last = parts[0], " ".join(parts[1:])
    t = playerid_lookup(last, first)
    if t.empty:
        raise ValueError(f"Could not find player: {name}")
    mlbam = int(t.iloc[0]["key_mlbam"])
    _player_cache[name] = mlbam
    return mlbam

def parse_count(count_str):
    if count_str is None:
        return 0, 0
    count_str = str(count_str).strip()
    b, s = count_str.split("-")
    return int(b), int(s)

def parse_runners(runners_str):
    # runners_str like "1;2" or "1" or "2;3" or None
    if runners_str is None:
        return 0, 0, 0
    r = str(runners_str).strip()
    if r == "" or r.lower() == "empty":
        return 0, 0, 0
    bases = set([x.strip() for x in r.split(";") if x.strip() != ""])
    return int("1" in bases), int("2" in bases), int("3" in bases)

def parse_score(score_str):
    # "5:4" => run_diff = batting - fielding = 1
    if score_str is None:
        return 0
    a, b = str(score_str).strip().split(":")
    return int(a) - int(b)

def parse_pitcher_blob(blob):
    # "Tarik Skubal, 3-1, 1;2" -> name, count, runners
    parts = [p.strip() for p in str(blob).split(",")]
    p_name = parts[0]
    p_count = parts[1] if len(parts) >= 2 else None
    p_runners = parts[2] if len(parts) >= 3 else None
    return p_name, p_count, p_runners

def _mode(series):
    return series.value_counts(dropna=True).index[0]

def _build_situation_samples(
    batter_id,
    pitcher_id,
    balls,
    strikes,
    runner_1b,
    runner_2b,
    runner_3b,
    run_diff,
    n=1000
):
    # strict pool: pitcher + count, backoff to pitcher-only
    pool = df[(df["pitcher"] == pitcher_id) & (df["balls"] == balls) & (df["strikes"] == strikes)]
    if len(pool) < 500:
        pool = df[df["pitcher"] == pitcher_id]
    if len(pool) < 500:
        raise ValueError("Not enough historical pitches for that pitcher in dataset")

    samp = pool.sample(n=min(n, len(pool)), replace=(len(pool) < n), random_state=42).copy()

    # lock in who
    samp["batter"] = batter_id
    samp["pitcher"] = pitcher_id

    # set situation
    samp["balls"] = balls
    samp["strikes"] = strikes
    samp["runner_1b"] = runner_1b
    samp["runner_2b"] = runner_2b
    samp["runner_3b"] = runner_3b
    samp["run_diff"] = run_diff

    # typical handedness
    b_hist = df[df["batter"] == batter_id]
    p_hist = df[df["pitcher"] == pitcher_id]
    if len(b_hist) > 0:
        samp["stand"] = _mode(b_hist["stand"])
    if len(p_hist) > 0:
        samp["p_throws"] = _mode(p_hist["p_throws"])

    # CRITICAL: prevent None/NaN issues during predict
    samp = samp.fillna(0)

    # ensure cat cols are strings (GPU safety)
    for c in ["pitcher","batter","pitch_type","stand","p_throws","zone"]:
        if c in samp.columns:
            samp[c] = samp[c].astype("string").fillna("NA")

    return samp[features]

def predict_matchup(
    batter_name,
    pitcher_blob,
    score=None,
    count=None,
    runners=None,
    n_sims=1000
):
    p_name, p_count, p_runners = parse_pitcher_blob(pitcher_blob)

    # pitcher blob overrides explicit kwargs
    if p_count is not None:
        count = p_count
    if p_runners is not None:
        runners = p_runners
    if count is None:
        count = "0-0"

    balls, strikes = parse_count(count)
    r1, r2, r3 = parse_runners(runners)
    rd = parse_score(score)

    batter_id = mlbam_id_from_name(batter_name)
    pitcher_id = mlbam_id_from_name(p_name)

    Xsim = _build_situation_samples(
        batter_id=batter_id,
        pitcher_id=pitcher_id,
        balls=balls,
        strikes=strikes,
        runner_1b=r1,
        runner_2b=r2,
        runner_3b=r3,
        run_diff=rd,
        n=n_sims
    )

    prob_mat = model.predict_proba(Xsim)
    avg_probs = prob_mat.mean(axis=0)

    out = pd.Series(avg_probs, index=model.classes_).sort_values(ascending=False)
    result = (out * 100).round(2).to_dict()

    return {
        "batter": batter_name,
        "pitcher": p_name,
        "count": f"{balls}-{strikes}",
        "runners": {"1b": r1, "2b": r2, "3b": r3},
        "run_diff": rd,
        "n_sims": n_sims,
        "probs_percent": result
    }

In [5]:
predict_matchup("Aaron Judge","Tarik Skubal, 3-1, 1;2","5:4")

{'batter': 'Aaron Judge',
 'pitcher': 'Tarik Skubal',
 'count': '3-1',
 'runners': {'1b': 1, '2b': 1, '3b': 0},
 'run_diff': 1,
 'n_sims': 1000,
 'probs_percent': {'walk': 50.85,
  'out': 19.34,
  'strikeout': 17.82,
  'single': 5.95,
  'hr': 3.48,
  'double': 2.51,
  'triple': 0.05}}

In [6]:
import numpy as np
import pandas as pd

def simulate_matchup(batter_name, pitcher_name, count="0-0", runners=None, score=None, sims=10000, random_state=42):
    res = predict_matchup(
        batter_name,
        f"{pitcher_name}, {count}" + (f", {runners}" if runners else ""),
        score,
        n_sims=min(2000, sims)
    )

    probs_pct = res["probs_percent"]
    outcomes = list(probs_pct.keys())
    probs = np.array(list(probs_pct.values()), dtype=float) / 100.0

    rng = np.random.default_rng(random_state)
    draws = rng.choice(outcomes, size=sims, p=probs)

    sim_pct = (pd.Series(draws).value_counts().reindex(outcomes, fill_value=0) / sims * 100).round(2).to_dict()

    return {
        **res,
        "sims": sims,
        "simulated_probs_percent": sim_pct
    }

In [7]:
simulate_matchup("Aaron Judge","Tarik Skubal",count="3-1",runners="1;2",score="5:4",sims=10000)

{'batter': 'Aaron Judge',
 'pitcher': 'Tarik Skubal',
 'count': '3-1',
 'runners': {'1b': 1, '2b': 1, '3b': 0},
 'run_diff': 1,
 'n_sims': 2000,
 'probs_percent': {'walk': 51.96,
  'out': 18.9,
  'strikeout': 17.4,
  'single': 5.83,
  'hr': 3.41,
  'double': 2.45,
  'triple': 0.05},
 'sims': 10000,
 'simulated_probs_percent': {'walk': 52.25,
  'out': 19.15,
  'strikeout': 17.17,
  'single': 5.51,
  'hr': 3.35,
  'double': 2.52,
  'triple': 0.05}}

In [ ]:
# =========================
# Interactive Matchup Menu (Jupyter/VS Code)
# =========================
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

# --- Inputs ---
batter_in = widgets.Text(
    value="Aaron Judge",
    description="Batter:",
    placeholder="First Last",
    layout=widgets.Layout(width="380px")
)

pitcher_in = widgets.Text(
    value="Tarik Skubal",
    description="Pitcher:",
    placeholder="First Last",
    layout=widgets.Layout(width="380px")
)

balls_dd = widgets.Dropdown(
    options=[0,1,2,3],
    value=0,
    description="Balls:"
)

strikes_dd = widgets.Dropdown(
    options=[0,1,2],
    value=0,
    description="Strikes:"
)

r1_cb = widgets.Checkbox(value=False, description="1B")
r2_cb = widgets.Checkbox(value=False, description="2B")
r3_cb = widgets.Checkbox(value=False, description="3B")

bat_score_in = widgets.IntText(value=0, description="Bat score:")
fld_score_in = widgets.IntText(value=0, description="Fld score:")

sims_sl = widgets.IntSlider(
    value=1000,
    min=200,
    max=5000,
    step=200,
    description="Sims:",
    continuous_update=False
)

submit_btn = widgets.Button(
    description="Submit",
    button_style="success",
    icon="check"
)

out_box = widgets.Output()

# --- Helpers ---
def _runners_str():
    bases = []
    if r1_cb.value: bases.append("1")
    if r2_cb.value: bases.append("2")
    if r3_cb.value: bases.append("3")
    return ";".join(bases) if bases else None

def _score_str():
    return f"{bat_score_in.value}:{fld_score_in.value}"

def on_submit(_):
    with out_box:
        clear_output()

        batter = batter_in.value.strip()
        pitcher = pitcher_in.value.strip()
        count = f"{balls_dd.value}-{strikes_dd.value}"
        runners = _runners_str()
        score = _score_str()
        n_sims = int(sims_sl.value)

        try:
            # use your existing function
            res = predict_matchup(
                batter,
                f"{pitcher}, {count}" + (f", {runners}" if runners else ""),
                score,
                n_sims=n_sims
            )

            probs = res["probs_percent"]
            df_out = (
                pd.DataFrame({"Outcome": list(probs.keys()), "Percent": list(probs.values())})
                .sort_values("Percent", ascending=False)
                .reset_index(drop=True)
            )

            print(f"{res['batter']} vs {res['pitcher']}")
            print(f"Count: {res['count']} | Runners: {res['runners']} | Run diff: {res['run_diff']} | Sims: {res['n_sims']}")
            display(df_out)

        except Exception as e:
            print("Error:", str(e))

submit_btn.on_click(on_submit)

# --- Layout ---
row1 = widgets.HBox([batter_in, pitcher_in])
row2 = widgets.HBox([balls_dd, strikes_dd, widgets.Label("Runners:"), r1_cb, r2_cb, r3_cb])
row3 = widgets.HBox([bat_score_in, fld_score_in, sims_sl, submit_btn])

ui = widgets.VBox([row1, row2, row3, out_box])
display(ui)

In [9]:
import pandas as pd
import numpy as np

raw = pd.read_parquet("statcast_2021_2025.parquet")
print("loaded raw:", raw.shape)

# pitch ordering column
sort_col = "pitch_number" if "pitch_number" in raw.columns else ("pitch_num" if "pitch_num" in raw.columns else None)
if sort_col is None:
    raise ValueError("Need 'pitch_number' or 'pitch_num' to order pitches within PA.")

needed = [
    "game_pk","at_bat_number",sort_col,
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"
]

# add bb_type if available (for ball in play type)
has_bb_type = "bb_type" in raw.columns
if has_bb_type:
    needed.append("bb_type")

missing = [c for c in needed if c not in raw.columns]
if missing:
    raise ValueError(f"Missing columns in parquet: {missing}")

tmp = raw[needed].copy()

# runners (current pitch state)
tmp["runner_1b"] = tmp["on_1b"].notna().astype(np.int8)
tmp["runner_2b"] = tmp["on_2b"].notna().astype(np.int8)
tmp["runner_3b"] = tmp["on_3b"].notna().astype(np.int8)
tmp.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# score context (batting - fielding)
tmp["run_diff"] = (tmp["bat_score"].fillna(0) - tmp["fld_score"].fillna(0)).astype(np.int16)
tmp.drop(columns=["bat_score","fld_score"], inplace=True)

# sort to get final pitch of each PA
tmp = tmp.sort_values(["game_pk","at_bat_number",sort_col])
last_pitch = tmp.groupby(["game_pk","at_bat_number"], as_index=False).tail(1).copy()

def map_pa_outcome(events, des):
    ev = None if pd.isna(events) else str(events)
    d = "" if pd.isna(des) else str(des).lower()

    if ev == "single": return "single"
    if ev == "double": return "double"
    if ev == "triple": return "triple"
    if ev == "home_run": return "hr"

    if ev in ["walk","intent_walk","hit_by_pitch"]:
        return "walk"

    if "strikeout" in d or ev in ["strikeout","strikeout_double_play"]:
        return "strikeout"

    if ev is not None:
        return "out"

    return "none"

def map_ball_type(bb_type, pa_outcome):
    # if no ball put in play (BB/K/etc), call it not_in_play
    if pa_outcome in ["walk","strikeout"]:
        return "not_in_play"
    if bb_type is None or pd.isna(bb_type):
        return "not_in_play"
    bt = str(bb_type).lower().strip()
    if bt in ["ground_ball","fly_ball","line_drive","popup"]:
        return bt
    return "not_in_play"

last_pitch["pa_outcome"] = [map_pa_outcome(e, d) for e, d in zip(last_pitch["events"], last_pitch["des"])]
last_pitch = last_pitch[last_pitch["pa_outcome"] != "none"].copy()

if has_bb_type:
    last_pitch["ball_type"] = [
        map_ball_type(bt, oc) for bt, oc in zip(last_pitch["bb_type"], last_pitch["pa_outcome"])
    ]
else:
    last_pitch["ball_type"] = "not_in_play"

last_pitch = last_pitch[["game_pk","at_bat_number","pa_outcome","ball_type"]].copy()

# attach to every pitch-state row in the PA
tmp = tmp.merge(last_pitch, on=["game_pk","at_bat_number"], how="inner")

# drop raw terminal fields (we now have targets)
drop_cols = ["events","des"]
if has_bb_type and "bb_type" in tmp.columns:
    drop_cols.append("bb_type")
tmp.drop(columns=drop_cols, inplace=True)

# features
features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

# numeric cleanup
for c in ["release_speed","release_spin_rate","pfx_x","pfx_z","zone"]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

tmp = tmp.dropna(subset=["pitch_type","stand","p_throws","release_speed","pfx_x","pfx_z","zone"]).copy()

df = tmp[features + ["pa_outcome","ball_type"]].copy()

print("modeling df:", df.shape)
print("\npa_outcome dist (%):")
print(df["pa_outcome"].value_counts(normalize=True).mul(100).round(2))
print("\nball_type dist (%):")
print(df["ball_type"].value_counts(normalize=True).mul(100).round(2))

loaded raw: (3846144, 119)
modeling df: (3741396, 20)

pa_outcome dist (%):
pa_outcome
out          39.80
strikeout    28.31
walk         13.03
single       12.16
double        3.72
hr            2.65
triple        0.33
Name: proportion, dtype: float64

ball_type dist (%):
ball_type
not_in_play    41.49
ground_ball    25.18
fly_ball       15.25
line_drive     13.94
popup           4.13
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# NOTE: assumes you already trained your main model named `model` on df["pa_outcome"]

Xb = df[features].copy()
yb = df["ball_type"].copy()

cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    Xb[c] = Xb[c].astype("string").fillna("NA")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.2, random_state=42, stratify=yb
)

model_balltype = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=800,
    learning_rate=0.06,
    depth=8,
    verbose=100,
    task_type="GPU",
    devices="0"
)

model_balltype.fit(
    Xb_train, yb_train,
    cat_features=cat_cols,
    eval_set=(Xb_test, yb_test),
    use_best_model=True
)

print("ball_type classes:", list(model_balltype.classes_))

0:	learn: 1.5802648	test: 1.5802497	best: 1.5802497 (0)	total: 117ms	remaining: 1m 33s
100:	learn: 1.3478857	test: 1.3470132	best: 1.3470132 (100)	total: 7.06s	remaining: 48.9s
200:	learn: 1.3420318	test: 1.3420104	best: 1.3420104 (200)	total: 13.8s	remaining: 41.2s
300:	learn: 1.3386429	test: 1.3394639	best: 1.3394639 (300)	total: 20.4s	remaining: 33.8s
400:	learn: 1.3360653	test: 1.3377895	best: 1.3377895 (400)	total: 27.1s	remaining: 26.9s
500:	learn: 1.3336989	test: 1.3363570	best: 1.3363570 (500)	total: 33.9s	remaining: 20.2s
600:	learn: 1.3315441	test: 1.3350907	best: 1.3350907 (600)	total: 40.5s	remaining: 13.4s
700:	learn: 1.3293461	test: 1.3338877	best: 1.3338877 (700)	total: 47.2s	remaining: 6.66s
799:	learn: 1.3273139	test: 1.3327961	best: 1.3327961 (799)	total: 53.8s	remaining: 0us
bestTest = 1.332796129
bestIteration = 799
ball_type classes: ['fly_ball', 'ground_ball', 'line_drive', 'not_in_play', 'popup']


In [11]:
def approx_rbi_prob(outcome_probs, r1, r2, r3):
    """
    outcome_probs: dict like {"out": 41.1, "single": 12.4, ...} in PERCENT units
    r1/r2/r3: 0/1 runners on base
    returns: RBI probability in percent (approx)
    """
    # no baserunners: only HR can produce RBI (1)
    runners = int(r1) + int(r2) + int(r3)
    if runners == 0:
        return float(outcome_probs.get("hr", 0.0))

    # heuristics: probability at least one run scores given the PA result
    # (simple, but works decently as a first pass)
    p = {k: v / 100.0 for k, v in outcome_probs.items()}

    # HR always scores at least 1 if anyone on (and also if empty, handled above)
    pr_hr = p.get("hr", 0.0)

    # walk: RBI only if bases loaded
    pr_walk_rbi = p.get("walk", 0.0) * (1.0 if (r1 and r2 and r3) else 0.0)

    # triple: almost always brings in at least 1 if anyone is on
    pr_triple_rbi = p.get("triple", 0.0) * 0.95

    # double: very likely to score runner from 2B/3B, sometimes from 1B
    if r2 or r3:
        dbl_rate = 0.85
    elif r1:
        dbl_rate = 0.45
    else:
        dbl_rate = 0.0
    pr_double_rbi = p.get("double", 0.0) * dbl_rate

    # single: likely scores runner from 3B, sometimes from 2B, rarely from 1B
    if r3:
        sng_rate = 0.80
    elif r2:
        sng_rate = 0.35
    elif r1:
        sng_rate = 0.05
    else:
        sng_rate = 0.0
    pr_single_rbi = p.get("single", 0.0) * sng_rate

    # outs: can produce RBI via sac fly / groundout (needs runner on 3B mostly)
    # small baseline chance
    out_rate = 0.06 if r3 else 0.01
    pr_out_rbi = p.get("out", 0.0) * out_rate

    pr_rbi = pr_hr + pr_walk_rbi + pr_triple_rbi + pr_double_rbi + pr_single_rbi + pr_out_rbi

    # cap at 1.0
    pr_rbi = min(pr_rbi, 1.0)
    return round(pr_rbi * 100.0, 2)

In [12]:
def predict_matchup(
    batter_name,
    pitcher_blob,
    score=None,
    count=None,
    runners=None,
    n_sims=1000
):
    p_name, p_count, p_runners = parse_pitcher_blob(pitcher_blob)

    if p_count is not None:
        count = p_count
    if p_runners is not None:
        runners = p_runners
    if count is None:
        count = "0-0"

    balls, strikes = parse_count(count)
    r1, r2, r3 = parse_runners(runners)
    rd = parse_score(score)

    batter_id = mlbam_id_from_name(batter_name)
    pitcher_id = mlbam_id_from_name(p_name)

    Xsim = _build_situation_samples(
        batter_id=batter_id,
        pitcher_id=pitcher_id,
        balls=balls,
        strikes=strikes,
        runner_1b=r1,
        runner_2b=r2,
        runner_3b=r3,
        run_diff=rd,
        n=n_sims
    )

    # --- PA outcome probs (your main model) ---
    prob_mat = model.predict_proba(Xsim)
    avg_probs = prob_mat.mean(axis=0)
    out_probs = pd.Series(avg_probs, index=model.classes_).sort_values(ascending=False)
    out_probs_pct = (out_probs * 100).round(2).to_dict()

    # --- ball type probs (second model) ---
    bt_mat = model_balltype.predict_proba(Xsim)
    bt_avg = bt_mat.mean(axis=0)
    bt_probs = pd.Series(bt_avg, index=model_balltype.classes_).sort_values(ascending=False)
    bt_probs_pct = (bt_probs * 100).round(2).to_dict()

    # --- RBI odds (Option B1 approximation) ---
    rbi_pct = approx_rbi_prob(out_probs_pct, r1, r2, r3)

    return {
        "batter": batter_name,
        "pitcher": p_name,
        "count": f"{balls}-{strikes}",
        "runners": {"1b": r1, "2b": r2, "3b": r3},
        "run_diff": rd,
        "n_sims": n_sims,
        "probs_percent": out_probs_pct,
        "ball_type_percent": bt_probs_pct,
        "rbi_percent": rbi_pct
    }

In [13]:
res = predict_matchup("Aaron Judge", "Tarik Skubal, 3-1, 1;2", "5:4")
res["rbi_percent"], res["ball_type_percent"], res["probs_percent"]

(7.94,
 {'not_in_play': 68.51,
  'fly_ball': 12.62,
  'ground_ball': 9.68,
  'line_drive': 6.54,
  'popup': 2.65},
 {'walk': 50.85,
  'out': 19.34,
  'strikeout': 17.82,
  'single': 5.95,
  'hr': 3.48,
  'double': 2.51,
  'triple': 0.05})